In [1]:
import os
import shutil
from tqdm import tqdm

collections = ["lastfm", "suno", "udio"]
src_root = "/workspace/dataset"
dst_root = "/workspace/dataset_2"

skipped = []
moved = []

for collection in collections:
    src_collection = os.path.join(src_root, collection)
    dst_collection = os.path.join(dst_root, collection)

    if not os.path.isdir(dst_collection):
        print(f"Destination collection folder missing: {dst_collection}. Skipping entire collection.")
        continue

    for song_id in tqdm(os.listdir(src_collection), desc=f"Processing {collection}"):
        src_id_folder = os.path.join(src_collection, song_id)
        dst_id_folder = os.path.join(dst_collection, song_id)

        if not os.path.isdir(src_id_folder):
            continue  # Not a folder in source

        if not os.path.isdir(dst_id_folder):
            skipped.append((collection, song_id, "no dst_id_folder"))
            continue  # Destination id folder does not exist: skip

        src_style_file = os.path.join(src_id_folder, f"{song_id}_style.json")
        dst_style_file = os.path.join(dst_id_folder, f"{song_id}_style.json")

        # Must exist in source and not in destination
        if not os.path.isfile(src_style_file):
            skipped.append((collection, song_id, "no src_style_file"))
            continue

        if os.path.exists(dst_style_file):
            skipped.append((collection, song_id, "dst_style_exists"))
            continue  # DO NOT overwrite

        try:
            # Double-check: never create folder, only move if destination folder exists
            if os.path.isdir(dst_id_folder):
                shutil.copy2(src_style_file, dst_style_file)
                moved.append((collection, song_id))
            else:
                skipped.append((collection, song_id, "no dst_id_folder (2nd check)"))
        except Exception as e:
            print(f"Error copying {src_style_file} to {dst_style_file}: {e}")
            skipped.append((collection, song_id, f"error: {e}"))

print(f"\nTotal files moved: {len(moved)}")
print(f"Total files skipped: {len(skipped)}")


Processing lastfm:   0%|          | 0/20003 [00:00<?, ?it/s]

Processing udio: 100%|██████████| 20158/20158 [00:00<00:00, 44479.86it/s]


Total files moved: 137
Total files skipped: 62308
